In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, NMF
from sklearn.manifold import TSNE, Isomap, MDS
import umap
import matplotlib.patches as mpatches
from scipy.stats import entropy
import pandas as pd
import numpy as np
import ast
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"

# Garante que a pasta de saída exista
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)

Project root: c:\Users\ccana\Documents\Doutorado\VISEMTracking
Raw data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\raw


In [6]:
df_symbolic = pd.read_csv("symbolic2.csv")
df_fatty_acids = pd.read_csv(DATA_RAW / "fatty_acids_serum_Train.csv")
df_acids_spermatoza = pd.read_csv(DATA_RAW / "fatty_acids_spermatoza_Train.csv")
df_participants = pd.read_csv(DATA_RAW / "participant_related_data_Train.csv")
df_semen_analysis = pd.read_csv(DATA_RAW / "semen_analysis_data_Train.csv")
df_sex_hormones = pd.read_csv(DATA_RAW / "sex_hormones_Train.csv")


In [27]:
df_symbolic['clusterSF'] = df_symbolic['cluster'].astype(str) + "_" + df_symbolic['phys_cluster'].astype(str)
df_merged = df_symbolic.copy()[['user_id','clusterSF']]
df_merged = df_merged.rename(columns={'user_id': 'ID'})
df_merged = pd.merge(df_merged, df_fatty_acids, on='ID', how='right')
df_merged = pd.merge(df_merged, df_acids_spermatoza, on='ID', how='right')
df_merged = pd.merge(df_merged, df_participants, on='ID', how='right')
df_merged = pd.merge(df_merged, df_semen_analysis, on='ID', how='right')
df_merged = pd.merge(df_merged, df_sex_hormones, on='ID', how='right')
print(f"Shape do dataframe mesclado: {df_merged.shape}")
df_merged.to_csv(DATA_RAW / "merged_cluster_hormos.csv", index=False)

Shape do dataframe mesclado: (807, 66)


In [38]:
import pandas as pd
import numpy as np
import math
import itertools
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt


# -----------------------------
# Utilitários
# -----------------------------
def _classify_numeric(series, thresh=0.95):
    s_num = pd.to_numeric(series, errors="coerce")
    total_non_na = series.notna().sum()
    numeric_non_na = s_num.notna().sum()
    is_num = (total_non_na > 0) and (numeric_non_na / total_non_na >= thresh)
    return is_num, s_num

def _shapiro_ok(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 3:
        return None
    if len(x) > 5000:
        rng = np.random.default_rng(0)
        x = rng.choice(x, size=5000, replace=False)
    return stats.shapiro(x).pvalue >= 0.05

def _eta_squared(F, df_between, df_within):
    denom = (F * df_between + df_within)
    return (F * df_between) / denom if denom and denom > 0 else np.nan

def _epsilon_squared(H, n, k):
    denom = (n - k)
    return (H - k + 1) / denom if denom and denom > 0 else np.nan

def _cramers_v(chi2, n, r, k):
    return np.sqrt((chi2 / n) / (min(r - 1, k - 1))) if n and min(r - 1, k - 1) > 0 else np.nan

def _dunn_posthoc(series, groups):
    """
    Dunn pós-teste (aprox. normal) com correção de empates.
    Retorna: cluster_a, cluster_b, z, p
    """
    tmp = pd.DataFrame({"x": series, "g": groups}).dropna()
    if tmp.empty:
        return pd.DataFrame()

    tmp["r"] = stats.rankdata(tmp["x"].to_numpy(), method="average")
    N = len(tmp)
    if N < 3:
        return pd.DataFrame()

    # tie correction
    _, counts = np.unique(tmp["x"].to_numpy(), return_counts=True)
    tie_term = np.sum(counts**3 - counts)
    C = 1 - tie_term / (N**3 - N) if (N**3 - N) != 0 else 1.0

    grp_stats = tmp.groupby("g").agg(n=("r", "size"), mean_rank=("r", "mean")).reset_index()
    grp_stats = grp_stats[grp_stats["n"] >= 2]
    g_levels = grp_stats["g"].tolist()
    if len(g_levels) < 2:
        return pd.DataFrame()

    mean_rank = dict(zip(grp_stats["g"], grp_stats["mean_rank"]))
    n = dict(zip(grp_stats["g"], grp_stats["n"]))

    var_factor = (N * (N + 1) / 12.0) * C

    out = []
    for a, b in itertools.combinations(g_levels, 2):
        se = math.sqrt(var_factor * (1 / n[a] + 1 / n[b]))
        if se == 0:
            z = 0.0
            p = 1.0
        else:
            z = (mean_rank[a] - mean_rank[b]) / se
            p = 2 * stats.norm.sf(abs(z))
        out.append({"cluster_a": a, "cluster_b": b, "z": z, "p_valor_bruto": p,
                    "n_a": int(n[a]), "n_b": int(n[b])})
    return pd.DataFrame(out)

def _games_howell(series, groups):
    """
    Games-Howell (aprox. t) para Welch ANOVA.
    Retorna: cluster_a, cluster_b, t, df, p
    """
    tmp = pd.DataFrame({"x": series, "g": groups}).dropna()
    if tmp.empty:
        return pd.DataFrame()

    g_levels = tmp["g"].unique()
    stats_g = {}
    for g in g_levels:
        vals = tmp.loc[tmp["g"] == g, "x"].to_numpy(dtype=float)
        n = len(vals)
        if n < 2:
            continue
        m = float(np.mean(vals))
        v = float(np.var(vals, ddof=1))
        stats_g[g] = (n, m, v)

    g_levels = list(stats_g.keys())
    if len(g_levels) < 2:
        return pd.DataFrame()

    out = []
    for a, b in itertools.combinations(g_levels, 2):
        n1, m1, v1 = stats_g[a]
        n2, m2, v2 = stats_g[b]
        se2 = v1 / n1 + v2 / n2
        if se2 <= 0:
            t = 0.0
            dfw = np.nan
            p = 1.0
        else:
            t = (m1 - m2) / math.sqrt(se2)
            num = se2**2
            den = (v1**2) / ((n1**2) * (n1 - 1)) + (v2**2) / ((n2**2) * (n2 - 1))
            dfw = num / den if den > 0 else np.nan
            p = 2 * stats.t.sf(abs(t), dfw) if np.isfinite(dfw) else 1.0

        out.append({"cluster_a": a, "cluster_b": b, "t": t, "df": dfw,
                    "p_valor_bruto": p, "n_a": int(n1), "n_b": int(n2),
                    "mean_diff_a_minus_b": m1 - m2})
    return pd.DataFrame(out)

def _mwu_effect_r(xa, xb, U):
    n1, n2 = len(xa), len(xb)
    mu = n1 * n2 / 2
    sigma = math.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    z = (U - mu) / sigma if sigma > 0 else 0.0
    r = z / math.sqrt(n1 + n2) if (n1 + n2) > 0 else np.nan
    return r


# -----------------------------
# Função principal (corrigida)
# -----------------------------
def analyze_clusters_impact_corrected(
    file_path: str,
    group_col: str = "clusterSF",
    id_cols=("ID",),
    alpha: float = 0.05,
    top_n: int = 5,
    output_prefix: str = "clusterSF"
):
    df = pd.read_csv(file_path)

    if group_col not in df.columns:
        raise ValueError(f"Coluna '{group_col}' não existe no CSV.")

    # remover NaN do cluster
    df = df[df[group_col].notna()].copy()
    df[group_col] = df[group_col].astype("string")

    clusters = sorted(df[group_col].unique().tolist())

    # descobrir atributos numéricos de forma robusta (coerção)
    numeric_cols = []
    numeric_series = {}
    for c in df.columns:
        if c == group_col or c in id_cols:
            continue
        is_num, s_num = _classify_numeric(df[c])
        if is_num:
            numeric_cols.append(c)
            numeric_series[c] = s_num

    # -----------------------------
    # 1) Testes globais por atributo (entre todos os clusters)
    # -----------------------------
    global_rows = []
    for col in numeric_cols:
        s = numeric_series[col]
        arrays = []
        lvls_used = []
        for cl in clusters:
            vals = s[df[group_col] == cl].to_numpy(dtype=float)
            vals = vals[~np.isnan(vals)]
            if len(vals) > 0:
                arrays.append(vals)
                lvls_used.append(cl)

        n_total = int(sum(len(a) for a in arrays))
        k = len(arrays)
        if k < 2 or n_total < 5:
            continue

        # normalidade por grupo (quando possível)
        normal_flags = []
        for a in arrays:
            ok = _shapiro_ok(a)
            if ok is not None:
                normal_flags.append(ok)
        all_normal = (len(normal_flags) > 0 and all(normal_flags))

        # homogeneidade de variâncias
        try:
            lev_p = stats.levene(*[a for a in arrays if len(a) >= 2]).pvalue
        except Exception:
            lev_p = np.nan

        if all_normal and (not np.isnan(lev_p)) and lev_p >= 0.05:
            F, p = stats.f_oneway(*arrays)
            test = "ANOVA"
            eff = _eta_squared(F, k - 1, n_total - k)
            statv = F
            eff_name = "eta2"
        elif all_normal:
            ares = anova_oneway(arrays, use_var="unequal")  # Welch
            test = "Welch ANOVA"
            statv = float(ares.statistic)
            p = float(ares.pvalue)
            eff = _eta_squared(statv, k - 1, n_total - k)  # aproximação
            eff_name = "eta2_aprox"
        else:
            H, p = stats.kruskal(*arrays)
            test = "Kruskal–Wallis"
            statv = H
            eff = _epsilon_squared(H, n_total, k)
            eff_name = "epsilon2"

        global_rows.append({
            "atributo": col,
            "teste_global": test,
            "estatistica": float(statv),
            "p_valor_bruto": float(p),
            "tamanho_efeito": float(eff),
            "nome_efeito": eff_name,
            "n_total": n_total,
            "k_clusters": k
        })

    global_df = pd.DataFrame(global_rows)
    if global_df.empty:
        raise RuntimeError("Nenhum atributo numérico analisável encontrado.")

    # FDR entre atributos (obrigatório)
    pvals = global_df["p_valor_bruto"].to_numpy(dtype=float)
    rej, p_adj, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")
    global_df["p_valor_ajustado_FDR"] = p_adj
    global_df["significativo_FDR"] = rej

    global_df = global_df.sort_values(["significativo_FDR", "p_valor_ajustado_FDR"], ascending=[False, True])

    out_global = f"{output_prefix}_global_tests.csv"
    global_df.to_csv(out_global, index=False)

    # -----------------------------
    # 2) Pós-testes pareados (somente para atributos globais significativos)
    # -----------------------------
    posthoc_rows = []
    sig_attrs = global_df.loc[global_df["significativo_FDR"], "atributo"].tolist()

    for attr in sig_attrs:
        test = global_df.loc[global_df["atributo"] == attr, "teste_global"].iloc[0]
        s = numeric_series[attr]
        g = df[group_col]

        if test == "Kruskal–Wallis":
            ddf = _dunn_posthoc(s, g)
            if ddf.empty:
                continue
            # ajuste por atributo (FDR dentro dos pares)
            pvals = ddf["p_valor_bruto"].to_numpy(dtype=float)
            _, p_adj, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")
            ddf["p_valor_ajustado_FDR"] = p_adj
            ddf["significativo"] = ddf["p_valor_ajustado_FDR"] < alpha
            for _, r in ddf.iterrows():
                a, b = r["cluster_a"], r["cluster_b"]
                med_a = np.nanmedian(s[g == a].to_numpy(dtype=float))
                med_b = np.nanmedian(s[g == b].to_numpy(dtype=float))
                posthoc_rows.append({
                    "atributo": attr,
                    "pos_teste": "Dunn + FDR(pares)",
                    "cluster_a": a,
                    "cluster_b": b,
                    "estatistica": float(r["z"]),
                    "p_bruto": float(r["p_valor_bruto"]),
                    "p_ajustado": float(r["p_valor_ajustado_FDR"]),
                    "significativo": bool(r["significativo"]),
                    "dif_mediana_a_minus_b": float(med_a - med_b),
                    "n_a": int(r["n_a"]),
                    "n_b": int(r["n_b"])
                })

        elif test == "Welch ANOVA":
            gh = _games_howell(s, g)
            if gh.empty:
                continue
            pvals = gh["p_valor_bruto"].to_numpy(dtype=float)
            _, p_adj, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")
            gh["p_valor_ajustado_FDR"] = p_adj
            gh["significativo"] = gh["p_valor_ajustado_FDR"] < alpha
            for _, r in gh.iterrows():
                a, b = r["cluster_a"], r["cluster_b"]
                med_a = np.nanmedian(s[g == a].to_numpy(dtype=float))
                med_b = np.nanmedian(s[g == b].to_numpy(dtype=float))
                posthoc_rows.append({
                    "atributo": attr,
                    "pos_teste": "Games–Howell + FDR(pares)",
                    "cluster_a": a,
                    "cluster_b": b,
                    "estatistica": float(r["t"]),
                    "p_bruto": float(r["p_valor_bruto"]),
                    "p_ajustado": float(r["p_valor_ajustado_FDR"]),
                    "significativo": bool(r["significativo"]),
                    "dif_mediana_a_minus_b": float(med_a - med_b),
                    "n_a": int(r["n_a"]),
                    "n_b": int(r["n_b"])
                })

        else:
            # ANOVA -> Tukey HSD (p-ajustado já controla família)
            tmp = pd.DataFrame({"x": s, "g": g}).dropna()
            if tmp["g"].nunique() < 2:
                continue
            tuk = pairwise_tukeyhsd(tmp["x"], tmp["g"], alpha=alpha)
            tdf = pd.DataFrame(tuk.summary().data[1:], columns=tuk.summary().data[0])
            for _, r in tdf.iterrows():
                a, b = r["group1"], r["group2"]
                med_a = np.nanmedian(s[g == a].to_numpy(dtype=float))
                med_b = np.nanmedian(s[g == b].to_numpy(dtype=float))
                posthoc_rows.append({
                    "atributo": attr,
                    "pos_teste": "Tukey HSD",
                    "cluster_a": a,
                    "cluster_b": b,
                    "estatistica": float(r["meandiff"]),
                    "p_bruto": np.nan,
                    "p_ajustado": float(r["p-adj"]),
                    "significativo": bool(r["reject"]),
                    "dif_mediana_a_minus_b": float(med_a - med_b),
                    "n_a": int((g == a).sum()),
                    "n_b": int((g == b).sum())
                })

    posthoc_df = pd.DataFrame(posthoc_rows)
    out_posthoc = f"{output_prefix}_posthoc_pairs.csv"
    posthoc_df.to_csv(out_posthoc, index=False)

    # -----------------------------
    # 3) One-vs-Rest (exploratório) com FDR por atributo
    # -----------------------------
    ovr_rows = []
    for col in numeric_cols:
        s = numeric_series[col]

        # MWU por cluster vs resto
        tmp_rows = []
        for cl in clusters:
            a = s[df[group_col] == cl].dropna().to_numpy(dtype=float)
            b = s[df[group_col] != cl].dropna().to_numpy(dtype=float)
            if len(a) < 3 or len(b) < 3:
                continue
            U, p = stats.mannwhitneyu(a, b, alternative="two-sided", method="auto")
            r = _mwu_effect_r(a, b, U)
            med_a = float(np.median(a))
            med_b = float(np.median(b))
            tmp_rows.append((cl, U, p, r, med_a, med_b))

        if not tmp_rows:
            continue

        # FDR dentro do atributo (por clusters)
        pvals = np.array([t[2] for t in tmp_rows], dtype=float)
        _, p_adj, _, _ = multipletests(pvals, alpha=alpha, method="fdr_bh")

        for (cl, U, p, r, med_a, med_b), pa in zip(tmp_rows, p_adj):
            ovr_rows.append({
                "atributo": col,
                "cluster": cl,
                "U": float(U),
                "p_bruto": float(p),
                "p_ajustado_FDR_por_atributo": float(pa),
                "significativo": bool(pa < alpha),
                "efeito_r": float(r),
                "mediana_cluster": float(med_a),
                "mediana_resto": float(med_b),
                "direcao": "Maior" if med_a > med_b else ("Menor" if med_a < med_b else "Igual")
            })

    ovr_df = pd.DataFrame(ovr_rows)
    out_ovr = f"{output_prefix}_one_vs_rest.csv"
    ovr_df.to_csv(out_ovr, index=False)

    # -----------------------------
    # 4) Heatmap cluster × atributo (mediana padronizada) dos atributos globais significativos
    # -----------------------------
    sig_num_attrs = sig_attrs
    if sig_num_attrs:
        med = pd.DataFrame(index=clusters, columns=sig_num_attrs, dtype=float)
        for attr in sig_num_attrs:
            s = numeric_series[attr]
            for cl in clusters:
                vals = s[df[group_col] == cl].to_numpy(dtype=float)
                vals = vals[~np.isnan(vals)]
                med.loc[cl, attr] = np.median(vals) if len(vals) else np.nan

        zmed = med.apply(
            lambda c: (c - np.nanmean(c)) / (np.nanstd(c, ddof=0) if np.nanstd(c, ddof=0) else np.nan),
            axis=0
        )

        out_med = f"{output_prefix}_cluster_by_attr_medians.csv"
        out_zmed = f"{output_prefix}_cluster_by_attr_medians_zscore.csv"
        med.to_csv(out_med)
        zmed.to_csv(out_zmed)

        plt.figure(figsize=(max(10, len(sig_num_attrs) * 0.25), max(6, len(clusters) * 0.35)))
        im = plt.imshow(zmed.to_numpy(dtype=float), aspect="auto", interpolation="nearest")
        plt.colorbar(im, label="Z-score da mediana (por atributo)")
        plt.yticks(range(len(clusters)), clusters)
        plt.xticks(range(len(sig_num_attrs)), sig_num_attrs, rotation=90, fontsize=7)
        plt.title("Heatmap: clusterSF × atributos significativos (mediana padronizada)")
        plt.tight_layout()
        out_heat = f"{output_prefix}_heatmap.png"
        plt.savefig(out_heat, dpi=200)
        plt.close()
    else:
        out_med = out_zmed = out_heat = None

    # -----------------------------
    # 5) Gráfico “Top N” por cluster (exploratório) — baseado em |efeito_r| e p-ajustado
    # -----------------------------
    if not ovr_df.empty:
        ovr_sig = ovr_df[ovr_df["significativo"]].copy()
        if not ovr_sig.empty:
            # escolhe top_n por cluster: menor p-ajustado, maior |r|
            ovr_sig["abs_r"] = ovr_sig["efeito_r"].abs()
            plot_rows = []
            for cl in clusters:
                sub = ovr_sig[ovr_sig["cluster"] == cl].sort_values(
                    ["p_ajustado_FDR_por_atributo", "abs_r"], ascending=[True, False]
                ).head(top_n)
                plot_rows.append(sub)
            top_plot = pd.concat(plot_rows, ignore_index=True) if plot_rows else pd.DataFrame()

            if not top_plot.empty:
                # um painel por cluster (matplotlib), simples e robusto
                ncols = 3
                nclusters = len(clusters)
                nrows = math.ceil(nclusters / ncols)
                plt.figure(figsize=(18, 4.5 * nrows))
                for i, cl in enumerate(clusters, 1):
                    ax = plt.subplot(nrows, ncols, i)
                    sub = top_plot[top_plot["cluster"] == cl].copy()
                    if sub.empty:
                        ax.axis("off")
                        ax.set_title(cl)
                        continue
                    # ordenar por diferença percentual (apenas para visual)
                    # aqui usamos diferença relativa de mediana como visual, mas inferência está em p/r
                    sub["diff_pct"] = np.where(
                        sub["mediana_resto"] == 0, np.nan,
                        (sub["mediana_cluster"] - sub["mediana_resto"]) / sub["mediana_resto"] * 100
                    )
                    sub = sub.sort_values("diff_pct", ascending=True)

                    ax.barh(sub["atributo"], sub["diff_pct"])
                    ax.axvline(0, linewidth=1)
                    ax.set_title(f"{cl}")
                    ax.set_xlabel("Variação da mediana (%) vs resto (apenas visual)")
                plt.tight_layout()
                out_top = f"{output_prefix}_top{top_n}_ovr.png"
                plt.savefig(out_top, dpi=200)
                plt.close()
            else:
                out_top = None
        else:
            out_top = None
    else:
        out_top = None

    print("Arquivos gerados:")
    print(" -", out_global)
    print(" -", out_posthoc)
    print(" -", out_ovr)
    if out_med:  print(" -", out_med)
    if out_zmed: print(" -", out_zmed)
    if out_heat: print(" -", out_heat)
    if out_top:  print(" -", out_top)

    return {
        "global": out_global,
        "posthoc": out_posthoc,
        "one_vs_rest": out_ovr,
        "medians": out_med,
        "zmedians": out_zmed,
        "heatmap": out_heat,
        "top_plot": out_top
    }


if __name__ == "__main__":
    analyze_clusters_impact_corrected(DATA_RAW / "merged_cluster_hormos.csv")

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\scipy\stats\_axis_nan_policy.py:531: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


Arquivos gerados:
 - clusterSF_global_tests.csv
 - clusterSF_posthoc_pairs.csv
 - clusterSF_one_vs_rest.csv
 - clusterSF_cluster_by_attr_medians.csv
 - clusterSF_cluster_by_attr_medians_zscore.csv
 - clusterSF_heatmap.png
 - clusterSF_top5_ovr.png
